# Guided Lab: First LangGraph Agent

This notebook builds a simple ReAct-style agent with:

- web search
- a calculator
- Groq as the model
- short-term memory via `InMemorySaver`

LangGraph short-term memory is thread-scoped and persisted through checkpoints, and Tavily integration requires `TAVILY_API_KEY` in the environment. The Deep Agents docs also show the general pattern of adding tools to an agent. 

## Learning goals

By the end of this notebook, you should be able to:

1. Load `GROQ_API_KEY` and `TAVILY_API_KEY` from `.env`.
2. Create a calculator tool.
3. Connect a Tavily web search tool.
4. Build a simple tool-using agent.
5. Run the agent with a `thread_id` so memory is preserved.

## 1) Install packages

In [42]:
%pip install -qU langgraph langchain langchain-core langchain-groq langchain-tavily python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2) Load API keys from `.env`

Create a `.env` file like this:

```env
GROQ_API_KEY=your_groq_api_key
TAVILY_API_KEY=your_tavily_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=first-langgraph-agent
```

Tavily’s LangChain integration docs say to set `TAVILY_API_KEY` in the environment. Short-term memory docs say thread-scoped state is managed through checkpoints. 

In [43]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true").lower() == "true"
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "first-langgraph-agent")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("TAVILY_API_KEY set:", bool(TAVILY_API_KEY))
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_TRACING:", LANGSMITH_TRACING)
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)

GROQ_API_KEY set: True
TAVILY_API_KEY set: True
LANGSMITH_API_KEY set: True
LANGSMITH_TRACING: True
LANGSMITH_PROJECT: lcel-groq-demo


## 3) Connect the Tavily search tool

This notebook follows your sample pattern: a Tavily search tool plus a calculator tool inside one agent.

If the new Tavily package is unavailable in your environment, the fallback import keeps the notebook compatible with the sample-style class name.

In [44]:
try:
    from langchain_tavily import TavilySearch
    search_tool = TavilySearch(max_results=2)
    search_tool.name = "web_search"
    search_tool.description = "Search the web for current information."
except Exception:
    from langchain_community.tools.tavily_search import TavilySearchResults
    search_tool = TavilySearchResults(max_results=2)

search_tool

TavilySearch(name='web_search', description='Search the web for current information.', max_results=2, api_wrapper=TavilySearchAPIWrapper(tavily_api_key=SecretStr('**********'), api_base_url=None))

## 4) Define a calculator tool

A calculator is useful for basic arithmetic and simple reasoning steps.

In [45]:
from langchain_core.tools import tool

@tool
def calculator(a: int, b: int) -> int:
    'Add two integers.'
    return a + b

calculator

StructuredTool(name='calculator', description='Add two integers.', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x000001E0E38CFE20>)

## 5) Create the Groq model

The notebook uses Groq for the agent model. You can set a model name in `.env` if you want to change it later.

In [53]:
import os
from langchain_groq import ChatGroq

GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_retries=3,                                 # network/429/5xx retries
    model_kwargs={"parallel_tool_calls": False},   # one tool call per turn
)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}} output_version=None profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x000001E0E39C3450> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001E0E39C3AD0> model_name='openai/gpt-oss-120b' temperature=1e-08 model_kwargs={'parallel_tool_calls': False} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None max_retries=3


## 6) Build the agent

The agent uses the tools and keeps thread-scoped memory with `InMemorySaver`.

LangGraph short-term memory is scoped to a thread and stored through checkpoints, which is why a `thread_id` matters when you invoke the agent. 

In [54]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tools = [search_tool, calculator]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. Use tools to find information and solve math problems. "
        "Use web search for current information and the calculator for arithmetic."
    ),
    checkpointer=InMemorySaver(),
)
print("Agent ready.")

Agent ready.


## 7) Run the sample request

This mirrors the sample flow you provided: a single user turn that asks for both web search and arithmetic.

In [55]:

import time
from langchain_core.messages import HumanMessage

def invoke_with_retry(agent, messages, config, max_attempts=3, base_delay=1.5):
    """
    Groq occasionally fails to format a tool call correctly (code 'tool_use_failed') —
    a generation glitch, not a problem with the input. The same call frequently
    succeeds on retry, and this isn't covered by ChatGroq's max_retries (that only
    retries network/429/5xx errors, not 400s).
    """
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return agent.invoke({"messages": messages}, config=config)
        except Exception as e:
            if "tool_use_failed" not in str(e):
                raise
            last_error = e
            print(f"[retry {attempt}/{max_attempts}] tool_use_failed, retrying...")
            time.sleep(base_delay * attempt)
    raise last_error

config = {"configurable": {"thread_id": "1"}}

response = invoke_with_retry(
    agent,
    [HumanMessage(content="What is the weather in SF and 2 + 2?")],
    config,
)

print(response["messages"][-1].content)

**Weather in San Francisco (as of the latest report on June 16 2026):**  
- **Condition:** Partly cloudy (night)  
- **Temperature:** 15.6 °C / 60.1 °F  
- **Humidity:** 86 %  
- **Wind:** 8.5 mph (13.7 kph) from the WSW (248°)  
- **Pressure:** 1014 mb (29.93 in)  
- **Visibility:** 16 km (9 mi)  
- **UV index:** 0  

**Math:**  
2 + 2 = 4.


## 8) Ask a follow-up in the same thread

Because the same `thread_id` is reused, the conversation stays in the same checkpointed thread.

In [56]:
follow_up = agent.invoke(
    {"messages": [("user", "And what was the arithmetic result again?")]},
    config=config,
)

follow_up

{'messages': [HumanMessage(content='What is the weather in SF and 2 + 2?', additional_kwargs={}, response_metadata={}, id='f5ac6739-ac13-43ff-af36-5d9aa9d4a666'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What is the weather in SF and 2 + 2?" Need to provide weather in San Francisco (SF) and compute 2+2. Use web search for weather. Use calculator for 2+2. Then answer.', 'tool_calls': [{'id': 'fc_627fc072-6806-44e7-8e1c-262849cda33b', 'function': {'arguments': '{"query":"current weather San Francisco","search_depth":"basic","time_range":"day"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 316, 'total_tokens': 415, 'completion_time': 0.212277309, 'completion_tokens_details': {'reasoning_tokens': 53}, 'prompt_time': 0.020177498, 'prompt_tokens_details': None, 'queue_time': 0.318511775, 'total_time': 0.232454807}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': '

## 9) Start a new thread

A new `thread_id` starts a separate conversation while still using the same agent and tools.

In [57]:
config_2 = {"configurable": {"thread_id": "2"}}

new_thread = agent.invoke(
    {"messages": [("user", "What is 15 + 27?")]},
    config=config_2,
)

new_thread

{'messages': [HumanMessage(content='What is 15 + 27?', additional_kwargs={}, response_metadata={}, id='b7c0bac2-f9fc-406d-9ba7-29644870deac'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We just need to compute 15+27 = 42. Use calculator tool.', 'tool_calls': [{'id': 'fc_97a93d47-2f33-4074-887b-e601cdecb989', 'function': {'arguments': '{"a":15,"b":27}', 'name': 'calculator'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 311, 'total_tokens': 363, 'completion_time': 0.108322521, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.013857391, 'prompt_tokens_details': None, 'queue_time': 0.287032051, 'total_time': 0.122179912}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8655ddce88', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ed453-fe2a-7ad0-b691-4332f47b1b8a-0', tool_calls=[{'name': 'calculator', 'arg

## 10) What this notebook demonstrates

- a ReAct-style tool-using agent
- live web search for current information
- calculator usage for math
- short-term memory through checkpointed threads
- a clean `.env` setup for `GROQ_API_KEY` and `TAVILY_API_KEY`

## Key takeaways

This notebook follows the LangGraph short-term memory model, where conversation state is thread-scoped and persisted through checkpoints, and it uses a simple agent-with-tools pattern similar to the LangChain and Deep Agents examples. Tavily’s integration expects `TAVILY_API_KEY`, so putting the key in `.env` keeps the setup clean. 

## References

- Deep research tool setup: https://docs.langchain.com/oss/python/deepagents/deep-research#add-tools
- Short-term memory: https://docs.langchain.com/oss/python/langgraph/short-term-memory
- Tavily integration: https://docs.langchain.com/oss/python/integrations/providers/tavily